In [1]:
import json

In [2]:
filename = "Raw_Intel_Reports.txt"

with open(filename, "r") as f:
    docs = json.load(f)
print(f"Loaded {len(docs)} documents from '{filename}'.")

Loaded 50 documents from 'Raw_Intel_Reports.txt'.


In [3]:
docs[0]

{'report_number': 1,
 'report_type': 'IMINT',
 'unit': 'Imagery Analysis Cell (Fict.)',
 'DTG': '2025-09-30T13:28Z',
 'primary_indicator': 'large-scale excavation pits appearing',
 'location_of_activity': 'Main Access Road',
 'observation_window': '12:27-13:20L',
 'executive_summary': 'Imagery analysis of EREBOS-3 reveals significant excavation activities along the Main Access Road, indicating potential expansion or construction projects. Observations between 12:27 and 13:20L on 2025-09-30 suggest a large-scale operation.',
 'key_judgments': ['With moderate-high confidence based on reliable satellite imagery, it is assessed that EREBOS-3 is undergoing substantial infrastructure development.',
  'The source reliability is high due to the clarity and consistency of the images obtained.',
  'Confidence in these judgments is further bolstered by the absence of any observable natural or environmental factors that could mimic such large-scale human activity.'],
 'details': 'During the observ

In [4]:
#list of strings
document_to_embed = []
#list of dictionaries
metadata = []
#list of strings
ids = []
text_to_embed = ["executive_summary", "key_judgements", "details", "source_description", "indicators", "recommended_tasks", "analyst_note"]

for report in docs:
    for field_name in text_to_embed:
        
        content = report.get(field_name)

        if not content:
            continue
        
        text_content = ""
        if isinstance(content, list):
            text_content = " ".join(content)
        else:
            text_content = content
        
        document_to_embed.append(text_content)
        
        metadata.append({
            'report_number': report.get('report_number'),
            'report_type': report.get('report_type'),
            'unit': report.get('unit'),
            'dtg': report.get('DTG'),
            'source_field': field_name
        })

        ids.append(f"report_{report.get('report_number')}_dictionary")

print("Entry #1")
print(f"ID: {ids[0]}")
print(f"METADATA: {metadata[0]}")
print(f"DOCUMENT: '{document_to_embed[0][:600]}...'")

Entry #1
ID: report_1_dictionary
METADATA: {'report_number': 1, 'report_type': 'IMINT', 'unit': 'Imagery Analysis Cell (Fict.)', 'dtg': '2025-09-30T13:28Z', 'source_field': 'executive_summary'}
DOCUMENT: 'Imagery analysis of EREBOS-3 reveals significant excavation activities along the Main Access Road, indicating potential expansion or construction projects. Observations between 12:27 and 13:20L on 2025-09-30 suggest a large-scale operation....'


In [5]:
from sentence_transformers import SentenceTransformer

c:\Users\noah.ross\Downloads\Strategy_Flow\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')  
embeddings = sentence_model.encode(document_to_embed, show_progress_bar=True)
print(f"Embeddings created successfully. Shape: {embeddings.shape}")
print(len(embeddings))
embeddings[0][:5]  # First 5 dimensions of the first embedding

Batches: 100%|██████████| 10/10 [00:01<00:00,  5.79it/s]

Embeddings created successfully. Shape: (300, 384)
300


array([-0.00066759, -0.00041645,  0.01367871,  0.02128864,  0.01862278],
      dtype=float32)

In [9]:
import pymupdf  
import re
from sentence_transformers import SentenceTransformer
import numpy as np
import redis
from redis.commands.search.field import (
    VectorField,
    TagField,
    TextField,
    NumericField
)
from redis.commands.search.index_definition import (
    IndexDefinition,
    IndexType
)

In [8]:
PDF_FILE_PATH = r"C:\Users\noah.ross\downloads\Strategy_Flow\Raw_Intel_Reports.pdf"
REDIS_HOST = "localhost"
REDIS_PORT = 6379
INDEX_NAME = "army_equipment_idx"
PREFIX = "doc:"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

In [11]:
print("\n--- Step 4: Loading data into Redis ---")

r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT)
r.ping()
print("Successfully connected to Redis.")

VECTOR_DIM = embeddings.shape[1]


--- Step 4: Loading data into Redis ---
Successfully connected to Redis.


In [14]:

    
schema = (
        TextField("content"), # The actual text chunk
        NumericField("report_number"), # For filtering by report
        TagField("report_type"),     # For filtering by type (e.g., "IMINT")
        TagField("source_field"),    # For filtering by section (e.g., "details")
        VectorField("vector", "HNSW", {"TYPE": "FLOAT32", "DIM": VECTOR_DIM, "DISTANCE_METRIC": "COSINE"}),
    )

definition = IndexDefinition(prefix=[PREFIX], index_type=IndexType.HASH)

#set up to delete existing index and create new one
r.ft(INDEX_NAME).dropindex(delete_documents=True)
print("Deleted existing index.")
r.ft(INDEX_NAME).create_index(fields=schema, definition=definition)
print(f"Index '{INDEX_NAME}' created successfully.")

    # --- MODIFIED PIPELINE ---
pipeline = r.pipeline()
for i, doc_text in enumerate(document_to_embed):
        vector_bytes = np.array(embeddings[i], dtype=np.float32).tobytes()
        
        # We now add all the new metadata to the hash
        item_data = {
            "content": doc_text,
            "report_number": metadata[i]["report_number"],
            "report_type": metadata[i]["report_type"],
            "source_field": metadata[i]["source_field"],
            "vector": vector_bytes
        }
        
        pipeline.hset(f"{PREFIX}{ids[i]}", mapping=item_data)
        
pipeline.execute()
    
print(f"\nThe Redis index '{INDEX_NAME}' now contains {len(ids)} documents.")



Deleted existing index.
Index 'army_equipment_idx' created successfully.

The Redis index 'army_equipment_idx' now contains 300 documents.
